# Tiền xử lý dữ liệu cho GAT-GRU

Notebook biến các run thô trong `results/` (Prometheus + Jaeger) thành chuỗi **đồ thị theo thời gian**:

$$\hat{Y}_{t+1:t+h} = f_{\text{GAT-GRU}}(\mathcal{G}_{t-w+1:t}), \qquad \mathcal{G}_\tau = (V, E, X_\tau, E^{attr}_\tau)$$

Toàn bộ logic (căn lưới, tạo feature, chia tập, chuẩn hóa, cắt cửa sổ) nằm trong `../modeling-common/forecast_data.py`. **Baseline LSTM dùng đúng module này**, nên hai mô hình nhận cùng dữ liệu, cùng feature nút, cùng cách chia và cùng chuẩn hóa. Khác biệt duy nhất là đồ thị.

**Quy trình mô hình:** `preprocess_gatgru` → `feature_selection_gatgru` → `train_gatgru` (Optuna + đánh giá) → `../lstm-load-testing/compare_lstm_gatgru`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COMMON = (Path('..') / 'modeling-common').resolve()   # code dùng chung cho GAT-GRU và LSTM
sys.path.insert(0, str(COMMON))
import forecast_data as fd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

## 1. Cấu hình
- Nguồn chưa có run thật thì notebook dùng **dữ liệu demo** (giả lập đúng định dạng file) và ghi ra thư mục `*_demo/`. Dữ liệu demo không dùng cho báo cáo.
- Cách chia train/val/test được lưu **một lần** vào file split và dùng chung cho mọi mô hình. Đây là điều kiện để so sánh LSTM và GAT-GRU trên **cùng các run test**.

**Chọn bộ dataset** bằng biến `DATASET` (đặt **giống nhau** ở mọi notebook của một lần thí nghiệm):

| `DATASET` | Nguồn | Kịch bản | Split | Thư mục đầu ra |
|---|---|---|---|---|
| `'load-testing'` | `load-testing/results/` (Locust, mô hình đóng) | normal, spike, bursty | `modeling-common/splits.json` | `processed/` |
| `'math'` | `math-load-testing/results/` (mô hình toán, luồng đến mở) | nhpp, mmpp, pareto, onoff | `modeling-common/splits_math.json` | `processed_math/` |

Hai bộ có split và thư mục kết quả riêng nên không ảnh hưởng nhau.

In [ ]:
DATASET = 'load-testing'          # 'load-testing' | 'math'
DATASETS = {
    'load-testing': {'label': 'load-testing', 'results': Path('results'), 'tag': '',
                      'scenarios': ('normal', 'spike', 'bursty'), 'test_scenarios': ['spike']},
    'math': {'label': 'math-load-testing', 'results': Path('../math-load-testing/results'), 'tag': '_math',
              'scenarios': ('nhpp', 'mmpp', 'pareto', 'onoff'), 'test_scenarios': ['mmpp']},
}
DS = DATASETS[DATASET]
RESULTS = {DS['label']: DS['results']}
INCLUDE_AUTOSCALERS = None        # None = mọi run; ví dụ ['none'] hoặc ['none', 'hpa']
CORRECT_COVERAGE = False          # True: nhân rps_in/call_rate với 1/jaeger_coverage của run
JAEGER_COVERAGE_MIN = 0.9
REBUILD_SPLITS = False            # chỉ đặt True TRƯỚC khi bắt đầu tune/đánh giá
SPLIT_KW = dict(mode='by_run', frac={'train': 0.70, 'val': 0.15, 'test': 0.15},
                test_scenarios=DS['test_scenarios'], seed=42)
DEFAULT_WINDOW = 12               # chỉ cho dataset mặc định; window thật được tune ở train notebook

# Chưa có run thật -> dữ liệu demo dùng chung (modeling-common/demo_results*), để chạy thử pipeline.
DEMO = not any(DS['results'].glob('*/run_*/meta.json'))
if DEMO:
    demo_root = COMMON / f"demo_results{DS['tag']}"
    if not demo_root.exists():
        fd.make_demo_results(demo_root, scenarios=DS['scenarios'])
    RESULTS = {DS['label']: demo_root}
SPLITS_PATH = COMMON / f"splits{DS['tag']}{'_demo' if DEMO else ''}.json"
OUT = Path(f"processed{DS['tag']}{'_demo' if DEMO else ''}")   # đầu vào cho feature_selection_gatgru / train_gatgru
OUT.mkdir(exist_ok=True)
print('DATASET =', DATASET, '| DEMO =', DEMO, '| RESULTS =', RESULTS, '| OUT =', OUT.resolve())

## 2. Nạp dữ liệu và kiểm tra chất lượng
`prometheus_coverage`: tỉ lệ ô (bước × service) có CPU. `jaeger_coverage`: request frontend đếm từ span / request Locust gửi. Dưới 0,9 nghĩa là Jaeger rơi span khi tải cao.

In [ ]:
runs = fd.load_runs(RESULTS)
assert runs, f'Không tìm thấy run nào trong {RESULTS}'
quality = fd.prepare_runs(runs)
if INCLUDE_AUTOSCALERS:
    runs = [r for r in runs if r['meta'].get('autoscaler', 'none') in INCLUDE_AUTOSCALERS]
bad = quality[(quality.prometheus_coverage < 0.9) | (quality.jaeger_coverage < JAEGER_COVERAGE_MIN)]
if len(bad):
    print('⚠️ Run có độ phủ thấp (cân nhắc loại hoặc chạy lại):')
    print(bad.to_string())
quality

## Danh sách feature ứng viên và lý do

Mọi feature phải **quan sát được khi triển khai thật** (controller đọc từ Prometheus/Jaeger). Số user của Locust **không** là feature, chỉ dùng để kiểm tra độ phủ. Đây là tập **ứng viên**; bước *feature selection* sẽ loại các feature không cần thiết.

| Nhóm | Feature | Nguồn / công thức | Mục tiêu |
|---|---|---|---|
| load | `rps_in` | Jaeger: số span `server` / 10s | Nhu cầu tải trực tiếp; **target** chính, không bị chặn bởi limit |
| load | `rps_in_delta` | $rps_t - rps_{t-1}$ | Xu hướng tức thời, giúp nhận ra đầu spike/burst sớm |
| load | `rps_per_replica` | `rps_in / replicas` | Tải trên mỗi pod, quyết định mức bão hòa |
| perf | `latency_p50_ms`, `latency_p95_ms` | phân vị thời lượng span server | Tín hiệu SLO; độ trễ lan ngược theo chuỗi gọi |
| perf | `error_rate` | lỗi / request (HTTP 5xx, gRPC ≠ 0) | Quá tải / lỗi dây chuyền |
| resource | `cpu_cores` | cAdvisor, chỉ container ứng dụng | Nhu cầu tài nguyên; **target** thứ hai |
| resource | `cpu_util_request` | `cpu_cores / cpu_request_cores` | Đúng đại lượng HPA dùng |
| resource | `cpu_throttle_ratio` | throttled / CFS periods | Bão hòa ở limit: CPU đứng yên nhưng latency tăng |
| resource | `cpu_sidecar_cores` | CPU `istio-proxy` | Tỉ lệ với lưu lượng qua Envoy; tín hiệu tải dự phòng |
| resource | `mem_mib` | working set | Áp lực bộ nhớ, cảnh báo OOM |
| resource | `net_rx_kBps`, `net_tx_kBps` | cAdvisor network | Traffic, gồm cả TCP không có span (redis) |
| capacity | `replicas` | KSM replicas available | Năng lực hiện tại, biến PPO điều khiển |
| capacity | `restarts_delta` | Δ restarts | Sự kiện bất ổn |

Feature **bắt buộc giữ** (`fd.FORCED_FEATURES`): `rps_in`, `cpu_cores` (target được dùng làm đầu vào tự hồi quy) và `replicas` (biến PPO điều khiển).

### Feature cạnh (chỉ GAT-GRU)
| Feature | Nguồn | Mục tiêu |
|---|---|---|
| `call_rate` | span `client` A→B / 10s | Trọng số phụ thuộc động, làm tín hiệu cho attention |
| `call_ratio` | `call_rate / rps_in(A)` | Hệ số fan-out, dùng để lan truyền tải xuống downstream |
| `edge_error_rate` | lỗi / lời gọi | Lỗi lan truyền dọc chuỗi gọi |
| `edge_latency_p95_ms` | p95 span client | Độ trễ upstream thấy ở downstream |

## 3. Service Graph
Tập cạnh = topology tĩnh ∪ cạnh quan sát được trong trace. `cartservice → redis-cart` là TCP, không có span, nên được giữ trong topology tĩnh với feature cạnh bằng 0.

In [ ]:
edges = fd.discover_edges(runs)
print(f'{len(edges)} cạnh ({len(fd.STATIC_EDGES)} tĩnh, {len(edges) - len(fd.STATIC_EDGES)} chỉ thấy trong trace)')
for r in runs:
    r['X'], r['E'], r['G'], r['tidy'] = fd.build_run_arrays(r, edges, CORRECT_COVERAGE)
tidy_all = pd.concat([r['tidy'] for r in runs], ignore_index=True)
print('X', runs[0]['X'].shape, '| E', runs[0]['E'].shape, '| G', runs[0]['G'].shape)
tidy_all.groupby('service')[['rps_in', 'cpu_cores', 'latency_p95_ms', 'replicas']].mean().round(3)

In [ ]:
mean_rate = np.concatenate([r['E'][..., 0] for r in runs]).mean(axis=0)
mean_rps = tidy_all.groupby('service').rps_in.mean().reindex(fd.SERVICES).fillna(0).to_numpy()
ang = np.linspace(0, 2 * np.pi, len(fd.SERVICES) - 1, endpoint=False)
pos = {'frontend': (0.0, 0.0)}
for a, s in zip(ang, [s for s in fd.SERVICES if s != 'frontend']):
    pos[s] = (np.cos(a), np.sin(a))
fig, ax = plt.subplots(figsize=(8, 8))
wgt = mean_rate / (mean_rate.max() or 1)
for (a, b), wi in zip(edges, wgt):
    ax.annotate('', xy=pos[b], xytext=pos[a], arrowprops=dict(arrowstyle='-|>', lw=0.6 + 5 * wi, color='C0',
                                                              alpha=0.35 + 0.6 * wi, shrinkA=14, shrinkB=14))
ax.scatter(*zip(*[pos[s] for s in fd.SERVICES]), s=300 + 2500 * mean_rps / (mean_rps.max() or 1),
           color='C1', zorder=3, alpha=0.85)
for s in fd.SERVICES:
    ax.text(pos[s][0], pos[s][1] - 0.13, s, ha='center', fontsize=8)
ax.set_title('Service Graph (độ dày cạnh ∝ call_rate, kích thước nút ∝ rps_in)')
ax.axis('off')
plt.show()

In [ ]:
scen = sorted({r['scenario'] for r in runs})
fig, axes = plt.subplots(3, len(scen), figsize=(5.5 * len(scen), 9), sharex='col', squeeze=False)
for j, sc in enumerate(scen):
    d = tidy_all[(tidy_all.scenario == sc) & (tidy_all.key == tidy_all[tidy_all.scenario == sc].key.min())]
    m = lambda x: x.t * fd.STEP_SEC / 60
    fe = d[d.service == 'frontend']
    axes[0, j].plot(m(fe), fe.rps_in, color='C3')
    axes[0, j].set_title(f'{sc.upper()}: frontend rps_in')
    for s in fd.SERVICES:
        ds = d[d.service == s]
        axes[1, j].plot(m(ds), ds.cpu_cores, lw=0.9, label=s)
        axes[2, j].plot(m(ds), ds.replicas, lw=0.9)
    axes[1, j].set_title('cpu_cores')
    axes[2, j].set_title('replicas')
    axes[2, j].set_xlabel('phút')
axes[1, -1].legend(fontsize=6, ncol=2)
fig.tight_layout()
plt.show()

## 4. Chia tập (dùng chung cho mọi mô hình)
Chia **theo run** (70/15/15 trong mỗi kịch bản), không theo cửa sổ. Các cửa sổ chồng lấn trong cùng run sẽ làm rò rỉ dữ liệu. Kịch bản có < 3 run thì chia theo thời gian bên trong run. File split tạo một lần và được tái sử dụng.

In [ ]:
splits = fd.load_or_create_splits(SPLITS_PATH, runs, rebuild=REBUILD_SPLITS, **SPLIT_KW)
runs = [r for r in runs if r['key'] in splits['runs']]
pd.DataFrame([{'scenario': r['scenario'], 'split': splits['runs'][r['key']]} for r in runs]) \
    .pivot_table(index='scenario', columns='split', aggfunc='size', fill_value=0)

## 5. Lưu dữ liệu
| File | Nội dung | Dùng cho |
|---|---|---|
| `runs_raw.npz` | Mảng **thô** từng run (mọi feature ứng viên, chưa chuẩn hóa, chưa cắt cửa sổ) + split | `feature_selection_gatgru`, `train_gatgru` (chọn feature/window trước, chuẩn hóa sau) |
| `gatgru_dataset.npz` | Cửa sổ mặc định (w = 12, mọi feature) đã chuẩn hóa | Thử nghiệm nhanh |
| `timeseries_raw.csv` | Bảng đơn vị gốc đã căn chỉnh | Môi trường PPO offline |

In [ ]:
fd.save_runs_npz(OUT / 'runs_raw.npz', runs, edges, {'splits': splits, 'demo': DEMO})
scalers = fd.fit_scalers(runs, splits, fd.NF, fd.EF, fd.GF)
windows = {s: fd.make_windows(runs, splits, s, DEFAULT_WINDOW, fd.HORIZON, fd.NF, fd.EF, scalers, fd.GF)
           for s in ('train', 'val', 'test')}
windows = {s: w for s, w in windows.items() if w is not None}
arrays = {'edge_index': np.array([[fd.SVC_IDX[a] for a, _ in edges], [fd.SVC_IDX[b] for _, b in edges]])}
for s, w in windows.items():
    arrays.update({f'{k}_{s}': (v.astype(str) if k == 'info' else v) for k, v in w.items()})
np.savez_compressed(OUT / 'gatgru_dataset.npz', **arrays)
(OUT / 'metadata.json').write_text(json.dumps({'edges': edges, 'scalers': scalers, 'window': DEFAULT_WINDOW,
                                               'horizon': fd.HORIZON, 'demo': DEMO}, indent=1, default=float))
tidy_all.to_csv(OUT / 'timeseries_raw.csv', index=False)
pd.DataFrame({s: {k: v.shape for k, v in w.items()} for s, w in windows.items()})

In [ ]:
import forecast_metrics as fm
w = windows.get('test') or windows.get('val')
if w is not None:
    d = {'Y_true': w['Y_raw'], 'Y_last': w['Y_last']}
    print('Baseline persistence (y_hat(t+k) = y(t)) trên', 'test' if 'test' in windows else 'val')
    display(fm.error_table(fm.persistence(d), w['Y_raw'], fd.TARGETS).round(4))